In [ ]:
import os,sys
from scipy import io
import scanpy as sc
import scvelo as scv
import cellrank as cr
from cellrank.kernels import CytoTRACEKernel
import anndata as ad
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import collections
import matplotlib
import warnings
from scipy.sparse import csr_matrix, diags
import matplotlib as mpl
mpl.rcParams.update({
    "font.family": "DejaVu Sans",   # e.g., "Arial", "Helvetica"
    "font.size": 6,
    "axes.titlesize": 6,
    "axes.labelsize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "svg.fonttype": "none",         # keep editable text in SVG
})




"""plt.rcParams.update(plt.rcParamsDefault)
rc={"axes.labelsize": 16, "xtick.labelsize": 12, "ytick.labelsize": 12,
    "figure.titleweight":"bold", #"font.size":14,
    "figure.figsize":(5.5,4.2), "font.weight":"regular", "legend.fontsize":10,
    'axes.labelpad':8, 'figure.dpi':300}
plt.rcParams.update(**rc)"""


MTpers=[30,30]
if MTpers[0]==MTpers[1]:
    MTper=MTpers[0]
else:
    MTper=str(MTpers[0]) + "_" + str(MTpers[1])
res=.1
res1=int(res*10**(len(str(res))-2))
dataName="Intestine"
sampleNames=["WT-HFD","LOF-HFD"]
sn="/combined"
for i in sampleNames:
  sn=sn+"_"+i
sn=sn+"_"+str(MTper)+"_"+str(res1)+"/"
print(sn)
mainDir = '/Users/krishangupta/Google Drive/My Drive/Desktop/Krishan_Gupta/Hong/' #yourDataAndCodeFolder
dataDir = mainDir+"loomFiles/RNA_Velocity/"+dataName+"/"+sn
plotDir = mainDir+'analysis/SCA_plots/'+dataName+"/"+sn+"/rnaVelocity/"
processedDataDir = mainDir+'analysis/processed_files/'+dataName+"/"+sn+"/rnaVelocity/"

if not os.path.exists(plotDir):
    os.mkdir(plotDir)
    
if not os.path.exists(processedDataDir):
    os.mkdir(processedDataDir)

# path='/Users/dongyuzhao/Library/CloudStorage/GoogleDrive-kulandaisamy.arulsamy@enders.tch.harvard.edu/My Drive/Dr_Hongs_data/Analysis/ScRNA_seq/Aorta_western_diet/DS/before_cutoff_backup/mebocost/'
dataNames=["WT-HFD","LOF-HFD"]
dataType='ABC_'
plotMap="umap"
conditionNames="conditions"
cellTypeNames=dataType+'subCellTypes'
cellTypeConditionNames=dataType+'Cond_SCT'
ext='.svg'
os.chdir(dataDir)
!pwd

In [ ]:
cell_meta = pd.read_csv(dataDir+"metadataF.csv")
cell_meta.loc[:,'cell_id']

In [ ]:
# load sparse matrix:
X = io.mmread(dataDir+"countsF.mtx")

# create anndata object
adata = ad.AnnData(
    X=X.transpose().tocsr()
)

# load cell metadata:
cell_meta = pd.read_csv(dataDir+"metadataF.csv")
cell_meta.index=cell_meta.loc[:,'cell_id']

# load gene names:
with open(dataDir+"gene_namesF.csv", 'r') as f:
    gene_names = f.read().splitlines()

# set anndata observations and index obs by barcodes, var by gene names
adata.obs = cell_meta
adata.obs.index = [bc[0:len(bc)-4] + '_' + cond for bc,cond in zip(adata.obs['barcode'].tolist(),adata.obs[conditionNames].tolist())]
adata.var.index = gene_names

# set pca and umap
adata.obsm['X_pca'] = np.vstack((adata.obs[dataType+'PCA_1'].to_numpy(), adata.obs[dataType+'PCA_2'].to_numpy())).T
adata.obsm['X_umap'] = np.vstack((adata.obs[dataType+'UMAP_1'].to_numpy(), adata.obs[dataType+'UMAP_2'].to_numpy())).T

# plot a UMAP colored by sampleID to test:
sc.pl.umap(adata, color=[cellTypeNames],size=5, frameon=False, save=cellTypeNames)

# save dataset as anndata format
adata.write(processedDataDir+'my_data.h5ad')

In [ ]:
ldata_WT_HFD_0 = scv.read(mainDir+"loomFiles/WT-HFD-Gut/"+'possorted_genome_bam_HJFBZ.loom', cache=True)
barcodes = [bc.split(':')[1] for bc in ldata_WT_HFD_0.obs.index.tolist()]
barcodes = [bc[0:len(bc)-1] + '_WT-HFD-Gut' for bc in barcodes]
ldata_WT_HFD_0.obs.index = barcodes
ldata_WT_HFD_0.var_names_make_unique()
print(ldata_WT_HFD_0)

ldata_LOF_HFD_0 = scv.read(mainDir+"loomFiles/LOF-HFD-Gut/"+'possorted_genome_bam_06T03.loom', cache=True)
barcodes = [bc.split(':')[1] for bc in ldata_LOF_HFD_0.obs.index.tolist()]
barcodes = [bc[0:len(bc)-1] + '_LOF-HFD-Gut' for bc in barcodes]
ldata_LOF_HFD_0.obs.index = barcodes
ldata_LOF_HFD_0.var_names_make_unique()
print(ldata_LOF_HFD_0)

ldata_WT_HFD_1 = scv.read(mainDir+"loomFiles/06_01_2023/WT-HFD-Gut/"+'possorted_genome_bam_L4PRW.loom', cache=True)
barcodes = [bc.split(':')[1] for bc in ldata_WT_HFD_1.obs.index.tolist()]
barcodes = [bc[0:len(bc)-1] + '_WT-HFD-Gut' for bc in barcodes]
ldata_WT_HFD_1.obs.index = barcodes
ldata_WT_HFD_1.var_names_make_unique()
print(ldata_WT_HFD_1)

ldata_LOF_HFD_1 = scv.read(mainDir+"loomFiles/06_01_2023/LOF-HFD-Gut/"+'possorted_genome_bam_LSIIV.loom', cache=True)
barcodes = [bc.split(':')[1] for bc in ldata_LOF_HFD_1.obs.index.tolist()]
barcodes = [bc[0:len(bc)-1] + '_LOF-HFD-Gut' for bc in barcodes]
ldata_LOF_HFD_1.obs.index = barcodes
ldata_LOF_HFD_1.var_names_make_unique()
print(ldata_LOF_HFD_1)

In [ ]:
adata = sc.read_h5ad(processedDataDir+'my_data.h5ad')
adata.obs_names = adata.obs_names.str.split("_", n=1).str[1]
adata.obs[cellTypeNames]

In [ ]:
adata = sc.read_h5ad(processedDataDir+'my_data.h5ad')
adata.obs_names = adata.obs_names.str.split("_", n=1).str[1]

WT_HFD_0=adata.obs['conditionsBatched'].index[adata.obs['conditionsBatched']=='WT-HFD_0']
print(WT_HFD_0.shape)
LOF_HFD_0=adata.obs['conditionsBatched'].index[adata.obs['conditionsBatched']=='LOF-HFD_0']
print(LOF_HFD_0.shape)
WT_HFD_1=adata.obs['conditionsBatched'].index[adata.obs['conditionsBatched']=='WT-HFD_1']
print(WT_HFD_1.shape)
LOF_HFD_1=adata.obs['conditionsBatched'].index[adata.obs['conditionsBatched']=='LOF-HFD_1']
print(LOF_HFD_1.shape)

adata_WT_HFD_0=adata[adata.obs['conditionsBatched']=='WT-HFD_0']
print(adata_WT_HFD_0.shape)
adata_LOF_HFD_0=adata[adata.obs['conditionsBatched']=='LOF-HFD_0']
print(adata_LOF_HFD_0.shape)
adata_WT_HFD_1=adata[adata.obs['conditionsBatched']=='WT-HFD_1']
print(adata_WT_HFD_1.shape)
adata_LOF_HFD_1=adata[adata.obs['conditionsBatched']=='LOF-HFD_1']
print(adata_LOF_HFD_1.shape)

adata_WT_HFD_0=scv.utils.merge(adata_WT_HFD_0, ldata_WT_HFD_0)
print(adata_WT_HFD_0.shape)
adata_LOF_HFD_0=scv.utils.merge(adata_LOF_HFD_0, ldata_LOF_HFD_0)
print(adata_LOF_HFD_0.shape)

adata_WT_HFD_1=scv.utils.merge(adata_WT_HFD_1, ldata_WT_HFD_1)
print(adata_WT_HFD_1.shape)
adata_LOF_HFD_1=scv.utils.merge(adata_LOF_HFD_1, ldata_LOF_HFD_1)
print(adata_LOF_HFD_1.shape)

adata = adata_WT_HFD_0.concatenate(adata_LOF_HFD_0,adata_WT_HFD_1,adata_LOF_HFD_1)
print(adata.shape)
adata

# save dataset as anndata format
adata.write(processedDataDir+'my_data_1.h5ad')

In [ ]:
cellTypes=['Cap LEC1','Cap LEC2','Cap LEC3',"Pre-Col LEC","Val LEC"]
initialState=cellTypes
terminalState=cellTypes

In [ ]:
import numpy as np
import random

np.random.seed(0)
random.seed(0)
CN=10000

initialState=cellTypes
terminalState=cellTypes
# n_states=9 #10

for dataName in ["WT-HFD","LOF-HFD"]:
    # reload dataset
    adata = sc.read_h5ad(processedDataDir+'my_data_1.h5ad')


    # dataName=dataNames[0]

    adata = adata[adata.obs[conditionNames].isin([dataName])] # filtering with conditions
    adata = adata[adata.obs[cellTypeNames].isin(cellTypes)] # filtering with cellTypes
    adata
    print(adata.shape)
    if adata.n_obs > CN:
        sampled_indices = np.random.RandomState(seed=0).choice(adata.obs_names, size=CN, replace=False)
        adata = adata[sampled_indices]

    # sc.pl.embedding(adata, basis=plotMap, color=cellTypeNames) # ploting with embeding and cellTypes
    scv.settings.verbosity = 3
    scv.settings.set_figure_params("scvelo")
    cr.settings.verbosity = 2
    warnings.simplefilter("ignore", category=UserWarning)
    # scv.pl.proportions(adata)

    # save dataset as anndata format
    adata.write(processedDataDir+'my_data_2.h5ad')

    ########################################################################
    # reload dataset
    adata = sc.read_h5ad(processedDataDir+'my_data_2.h5ad')

    scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000, subset_highly_variable=True)
    sc.tl.pca(adata)
    sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30, random_state=0)
    scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

    scv.tl.recover_dynamics(adata, n_jobs=8)
    scv.tl.velocity(adata, mode="dynamical")

    vk = cr.kernels.VelocityKernel(adata)
    vk.compute_transition_matrix()
    vk.write_to_adata()
    adata.write(processedDataDir+"_"+dataType+'_RNA_velocity.h5ad')

    ########################################################################
    adata = sc.read_h5ad(processedDataDir+"_"+dataType+'_RNA_velocity.h5ad')
    vk = cr.kernels.VelocityKernel.from_adata(adata, key="T_fwd")
    print(vk)

    ck = cr.kernels.ConnectivityKernel(adata)
    ck.compute_transition_matrix()
    ck.plot_projection(color=cellTypeNames) # ploting with cellTypes

    combined_kernel = .8 * vk + .2 * ck
    vk.plot_projection(color=cellTypeNames) # ploting with cellTypes
    # vk.plot_random_walks(start_ixs={cellTypeNames: initialState}, max_iter=200, seed=0) # initial state
    vk.write_to_adata()
    adata.write(processedDataDir+"_"+dataType+'_RNA_combined.h5ad')

    ########################################################################

    for n_states in range(3, 8, 1):
        try:
            print(n_states, dataName)
            adata = sc.read_h5ad(processedDataDir+"_"+dataType+'_RNA_combined.h5ad')
            vk = cr.kernels.VelocityKernel.from_adata(adata, key="T_fwd")
            # print(vk)

            g = cr.estimators.GPCCA(vk)
            # print(g)

            g.fit(cluster_key=cellTypeNames, n_states=[n_states, n_states])  # for HFR [12, 12] # for DKO [4, 3]
            # g.plot_macrostates(which="all", discrete=True, legend_loc="right", s=100)


            g.predict_initial_states(allow_overlap=True)
            # g.plot_macrostates(which="initial", legend_loc="right", s=100)

            terminalState=g.macrostates.values.unique()
            terminalState=[i for i in terminalState if i==i]
            # terminalState=['HC2 Alb hi', 'HC Alb Hnf4a hi', 'HC Hnf4a hi'] # only for WT, but not nesassary to do
            # print(terminalState)


            # g.predict_terminal_states()
            g.set_terminal_states(states=terminalState,allow_overlap=True)
            # g.plot_macrostates(which="terminal", legend_loc="right", s=100)

            # g.plot_macrostates(which="terminal", discrete=False)


            g.compute_fate_probabilities()
            # g.plot_fate_probabilities(same_plot=False)
            # g.plot_coarse_T()


            lineages=terminalState

            """for i in lineages:
                cr.pl.aggregate_fate_probabilities(
                adata,
                mode="violin",
                lineages=i,
                cluster_key=cellTypeNames,
                # clusters=fev_states,
                save=plotDir+"_"+dataType+'_cellrank_Violin.svg',
                figsize=(5, 4),
              dpi=400
            )"""

            all_terminal_sites_prob=pd.DataFrame(g.fate_probabilities)
            cell_index=pd.DataFrame(adata.obs)
            cell_index=cell_index[[cellTypeNames]]
            all_terminal_sites_prob.index=cell_index.index
            df2=pd.concat([all_terminal_sites_prob,cell_index],axis=1)
            terminalState.append(df2.columns[-1])
            df2.columns=terminalState
            ########################################################################
            #######################################################################
            cellTypes=sorted(list(df2[cellTypeNames].unique()))
            df1=df2.copy()
            for ct in cellTypes:
                # print(ct)
                matches = [item for item in df1.columns if ct in item]
                if len(matches)==0:
                    df1[ct]=0
                if len(matches)>1:
                    df1[ct]=0
                    for sct in matches:
                        df1[ct]=df1[ct]+df1[sct]
                    df1 = df1.drop(columns=matches)
            print(df1.columns)

            df=df1.groupby(cellTypeNames).mean().round(3)
            # print(df.sum(axis=1))
            # df.columns=terminalState

            # print(df)

            orderCells=cellTypes
            df=df.loc[orderCells[::-1],orderCells] # terminal state


            plt.figure(figsize=(12, 6))
            heatmap = sns.heatmap(df, annot=True, cmap="YlGnBu", fmt=".3f")

            # Setting the labels
            plt.title('Probabilities')
            plt.xlabel('Terminal State')
            plt.ylabel('Initial State')
            plt.savefig(plotDir+dataName+"_"+str(n_states)+"_heatmap.svg", format="svg")
            plt.show()

            """df_melted = pd.melt(df1, id_vars=terminalState[-1], var_name='Terminal state', value_name='Probability')
            category_order = cellTypes # initial State
            df_melted['Terminal state'] = pd.Categorical(df_melted['Terminal state'], categories=category_order, ordered=True)

            print("\nMelted DataFrame:")
            print(df_melted)

            plt.figure(figsize=(10, 4))  # Set the figure size
            sns.violinplot(x=terminalState[-1], y='Probability', hue='Terminal state', data=df_melted,
                        scale='width',  width=0.8, dodge=True)

            # Add title and labels
            plt.title('Box Plot for Each Column with Hue')
            plt.xlabel('Variable')
            plt.ylabel('Value')
            plt.legend(title='', bbox_to_anchor=(.55, .65), loc='center', ncol=1, frameon=True)
            plt.savefig(plotDir+dataName+"_violinplot.svg", format="pdf")
            # Display the plot
            plt.show()"""
        except Exception as e:
            print(f"Error in iteration {str(n_states)}, {dataName}: {e}")